In [20]:
# ==========================================
# 1 INSTALL LIBRARIES (RUN ONCE)
# ==========================================
# !pip install ml-dtypes==0.5.1 tensorflow==2.17.1 librosa soundfile scikit-learn pandas -q


# ==========================================
# 2 IMPORT LIBRARIES
# ==========================================
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import pickle
import zipfile
import warnings
import librosa
import numpy as np
import tensorflow as tf
import soundfile as sf

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report

warnings.filterwarnings('ignore')
print("✅ TensorFlow:", tf.__version__)


# ==========================================
# 3 EXTRACT DATASET
# ==========================================
zip_path = "/content/archive (1).zip"
extract_path = "/content/dataset"

if not os.path.exists(zip_path):
    raise FileNotFoundError("❌ Upload 'archive (1).zip' first!")

os.makedirs(extract_path, exist_ok=True)

# Extract only once
if not any(f.endswith(".wav") for _, _, files in os.walk(extract_path) for f in files):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print("✅ Dataset Extracted")
else:
    print("✅ Dataset already extracted")


# ==========================================
# 4 DEBUG FILES (IMPORTANT)
# ==========================================
print("\n🔍 Sample dataset files:")
count = 0
for root, _, files in os.walk(extract_path):
    for f in files:
        print(os.path.join(root, f))
        count += 1
        if count >= 10:
            break
    if count >= 10:
        break


# ==========================================
# 5 FEATURE EXTRACTION (FIXED)
# ==========================================
def extract_features(file_path):
    try:
        audio, sr = sf.read(file_path)

        # convert stereo → mono
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)

        # trim to 1.5 sec
        audio = audio[:int(sr * 1.5)]

        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        return np.mean(mfcc.T, axis=0)

    except Exception as e:
        print(f"❌ Skipped: {file_path}")
        return None


# ==========================================
# 6 LOAD DATASET
# ==========================================
X, y = [], []
MAX_FILES = 1200
errors = 0

for root, _, files in os.walk(extract_path):
    audio_files = [f for f in files if f.lower().endswith((".wav", ".mp3"))]

    if not audio_files:
        continue

    label = os.path.basename(root)

    for file in audio_files:
        if len(X) >= MAX_FILES:
            break

        path = os.path.join(root, file)
        features = extract_features(path)

        if features is not None:
            X.append(features)
            y.append(label)
        else:
            errors += 1

    if len(X) >= MAX_FILES:
        break

X = np.array(X, dtype=np.float32)
y = np.array(y)

print("\n✅ Samples Loaded:", len(X))
print("⚠️ Errors Skipped:", errors)
print("📊 Classes:", set(y))

if len(X) == 0:
    raise ValueError("❌ Dataset issue: No valid audio files!")


# ==========================================
# 7 PREPROCESSING
# ==========================================
scaler = StandardScaler()
X = scaler.fit_transform(X)

encoder = LabelEncoder()
y = encoder.fit_transform(y)

num_classes = len(encoder.classes_)
print("✅ Classes:", encoder.classes_)


# ==========================================
# 8 SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


# ==========================================
# 9 MODEL
# ==========================================
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(40,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


# ==========================================
# 10 EPOCH-WISE TRAINING
# ==========================================
EPOCHS = 10

for epoch in range(EPOCHS):
    print(f"\n🔥 Epoch {epoch+1}/{EPOCHS}")

    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=1,
        batch_size=32,
        verbose=1
    )

    acc = history.history['accuracy'][0]
    val = history.history['val_accuracy'][0]

    print(f"📊 Train: {acc:.4f} | Val: {val:.4f}")


# ==========================================
# 11 EVALUATION
# ==========================================
loss, acc = model.evaluate(X_test, y_test)
print(f"\n✅ Final Accuracy: {acc:.4f}")


# ==========================================
# 12 SAVE MODEL
# ==========================================
model.save("audio_model_fast.keras")

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("✅ Model + Files Saved")


# ==========================================
# 13 DOWNLOAD (COLAB)
# ==========================================
from google.colab import files

files.download("audio_model_fast.keras")
files.download("scaler.pkl")
files.download("label_encoder.pkl")

✅ TensorFlow: 2.17.1
✅ Dataset already extracted

🔍 Sample dataset files:
/content/dataset/M16/5 (49).wav
/content/dataset/M16/5 (80).wav
/content/dataset/M16/5 (14).wav
/content/dataset/M16/5 (46).wav
/content/dataset/M16/5 (24).wav
/content/dataset/M16/5 (10).wav
/content/dataset/M16/5 (1).wav
/content/dataset/M16/5 (12).wav
/content/dataset/M16/5 (97).wav
/content/dataset/M16/5 (59).wav

✅ Samples Loaded: 851
⚠️ Errors Skipped: 0
📊 Classes: {'MG-42', 'AK-12', 'M249', 'IMI Desert Eagle', 'AK-47', 'M4', 'MP5', 'M16', 'Zastava M92'}
✅ Classes: ['AK-12' 'AK-47' 'IMI Desert Eagle' 'M16' 'M249' 'M4' 'MG-42' 'MP5'
 'Zastava M92']

🔥 Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.3368 - loss: 1.8913 - val_accuracy: 0.4269 - val_loss: 1.5272
📊 Train: 0.3368 | Val: 0.4269

🔥 Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5309 - loss: 1.3413 - val_accuracy: 0.5380 - val_loss: 1.2708
📊 Train: 0.5309 | Val: 0.5380

🔥 Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
# ==========================================
# 14 GRADIO UI
# ==========================================
!pip install gradio -q

import gradio as gr
import numpy as np
import librosa
import soundfile as sf
import pickle
import tensorflow as tf


# ==========================================
# LOAD SAVED FILES
# ==========================================
model = tf.keras.models.load_model("audio_model_fast.keras")

with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("label_encoder.pkl", "rb") as f:
    encoder = pickle.load(f)


# ==========================================
# PREDICTION FUNCTION
# ==========================================
def predict_audio(file):

    try:
        audio, sr = sf.read(file)

        # convert stereo → mono
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)

        # trim
        audio = audio[:int(sr * 1.5)]

        # extract MFCC
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        features = np.mean(mfcc.T, axis=0)

        # scale
        features = scaler.transform([features])

        # predict
        pred = model.predict(features)
        class_index = np.argmax(pred)

        return f"🎯 Predicted Class: {encoder.inverse_transform([class_index])[0]}"

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ==========================================
# CREATE UI
# ==========================================
interface = gr.Interface(
    fn=predict_audio,
    inputs=gr.Audio(type="filepath"),
    outputs="text",
    title="🎤 Audio Classification System",
    description="Upload an audio file to classify it using Deep Learning"
)


# ==========================================
# LAUNCH APP
# ==========================================
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1f42227aaf31f1b541.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
